# DEAI-opdrachten Lineaire Regressie

### 2. Inlezen

In [68]:
import pandas as pd
from sklearn.model_selection import train_test_split
#from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler

In [69]:
df = pd.read_excel("AmesHousing.xlsx", sheet_name="AmesHousing")

# check
df.head()

# Laat de colom naamen zien
df.columns

Index(['ID', 'SalePrice', 'Garage', 'Overall Qual', 'Gr Liv Area',
       'Total Bsmt SF', 'Lot Area', 'Year Built', 'Full Bath', 'Bedroom AbvGr',
       'Neighborhood', 'House Style'],
      dtype='str')

### 3. features en target

In [73]:
# Check of ze egt bestaan
df[["Gr Liv Area", "Overall Qual", "Neighborhood"]].head()
df[["Gr Liv Area", "Overall Qual", "Year Built", "Total Bsmt SF", "Neighborhood"]].head()

,Gr Liv Area,Overall Qual,Year Built,Total Bsmt SF,Neighborhood
0,1656,6,1960,1080.0,NAmes
1,896,5,1961,882.0,NAmes
2,1329,6,1958,1329.0,NAmes
3,2110,7,1968,2110.0,NAmes
4,1629,5,1997,928.0,Gilbert


### 4.A One-hot

In [74]:
# Zet tekst om in true en false zodat er mee gerekend kan worden
df_encoded = pd.get_dummies(df, columns=["Neighborhood"])

df_encoded.head()

,ID,SalePrice,Garage,Overall Qual,Gr Liv Area,Total Bsmt SF,Lot Area,Year Built,Full Bath,Bedroom AbvGr,...,Neighborhood_NoRidge,Neighborhood_NridgHt,Neighborhood_OldTown,Neighborhood_SWISU,Neighborhood_Sawyer,Neighborhood_SawyerW,Neighborhood_Somerst,Neighborhood_StoneBr,Neighborhood_Timber,Neighborhood_Veenker
0,1,215000,yes,6,1656,1080.0,31770,1960,1,3,...,False,False,False,False,False,False,False,False,False,False
1,2,105000,yes,5,896,882.0,11622,1961,1,2,...,False,False,False,False,False,False,False,False,False,False
2,3,172000,yes,6,1329,1329.0,14267,1958,1,3,...,False,False,False,False,False,False,False,False,False,False
3,4,244000,yes,7,2110,2110.0,11160,1968,2,3,...,False,False,False,False,False,False,False,False,False,False
4,5,189900,yes,5,1629,928.0,13830,1997,2,3,...,False,False,False,False,False,False,False,False,False,False


### 4.B Split horizontaal en verticaal

In [84]:
# X = wat je gebruikt om te voorspelen
# col for col in .... pakt alle kolommen van Neighborhood die uit de one_hot kwamen
# De onderste code woort voor test 1, 2 en 3 gebruikt
# X = df_encoded[["Gr Liv Area", "Overall Qual"] + [col for col in df_encoded.columns if "Neighborhood_" in col]]

# deze woort gebruikt voor test 4
X = df_encoded[
    ["Gr Liv Area", "Overall Qual", "Year Built"] +
    [col for col in df_encoded.columns if "Neighborhood_" in col]
]
# Y = output
y = df_encoded["SalePrice"]

### Dataset Splitsen

In [85]:
# train set = model leren & Test set = eerlijk testen
# Anders leert de model alleen maar uit zijn hoofd
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Dit is voor de nieuwe maarnier en zorg dat R2 niet zeer laag staan (het lost dat probleem dus op)
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### 5. Model maken en trainen

In [86]:
# zo had ik het eerst:
# fit_intercept=True = model begint niet bij 0 
# # positive=False = gewichten mogen negatief zijn 
# test 1: model = LinearRegression(fit_intercept=True, positive=False) 

# aangepast:
# SGDRegressor() = leert stap voor stap
# max_iter = epochs
# eta0 = Learning rate
# test 2: model = SGDRegressor(max_iter=1000, eta0=0.01, random_state=42)
# de ondeste code woord voor test 3 en 4 gebruikt
model = SGDRegressor(max_iter=2000, eta0=0.001, random_state=42)
model.fit(X_train, y_train)

,"loss loss: str, default='squared_error'The loss function to be used. The possible values are 'squared_error','huber', 'epsilon_insensitive', or 'squared_epsilon_insensitive'The 'squared_error' refers to the ordinary least squares fit.'huber' modifies 'squared_error' to focus less on getting outlierscorrect by switching from squared to linear loss past a distance ofepsilon. 'epsilon_insensitive' ignores errors less than epsilon and islinear past that; this is the loss function used in SVR.'squared_epsilon_insensitive' is the same but becomes squared loss pasta tolerance of epsilon.More details about the losses formulas can be found in the:ref:`User Guide `.",'squared_error'
,"penalty penalty: {'l2', 'l1', 'elasticnet', None}, default='l2'The penalty (aka regularization term) to be used. Defaults to 'l2'which is the standard regularizer for linear SVM models. 'l1' and'elasticnet' might bring sparsity to the model (feature selection)not achievable with 'l2'. No penalty is added when set to `None`.You can see a visualisation of the penalties in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_penalties.py`.",'l2'
,"alpha alpha: float, default=0.0001Constant that multiplies the regularization term. The higher thevalue, the stronger the regularization. Also used to compute thelearning rate when `learning_rate` is set to 'optimal'.Values must be in the range `[0.0, inf)`.",0.0001
,"l1_ratio l1_ratio: float, default=0.15The Elastic Net mixing parameter, with 0 <= l1_ratio <= 1.l1_ratio=0 corresponds to L2 penalty, l1_ratio=1 to L1.Only used if `penalty` is 'elasticnet'.Values must be in the range `[0.0, 1.0]` or can be `None` if`penalty` is not `elasticnet`... versionchanged:: 1.7 `l1_ratio` can be `None` when `penalty` is not ""elasticnet"".",0.15
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If False, thedata is assumed to be already centered.",True
,"max_iter max_iter: int, default=1000The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the ``fit`` method, and not the:meth:`partial_fit` method.Values must be in the range `[1, inf)`... versionadded:: 0.19",2000
,"tol tol: float or None, default=1e-3The stopping criterion. If it is not None, training will stopwhen (loss > best_loss - tol) for ``n_iter_no_change`` consecutiveepochs.Convergence is checked against the training loss or thevalidation loss depending on the `early_stopping` parameter.Values must be in the range `[0.0, inf)`... versionadded:: 0.19",0.001
,"shuffle shuffle: bool, default=TrueWhether or not the training data should be shuffled after each epoch.",True
,"verbose verbose: int, default=0The verbosity level.Values must be in the range `[0, inf)`.",0
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-insensitive loss functions; only if `loss` is'huber', 'epsilon_insensitive', or 'squared_epsilon_insensitive'.For 'huber', determines the threshold at which it becomes lessimportant to get the prediction exactly right.For epsilon-insensitive, any differences between the current predictionand the correct label are ignored if they are less than this threshold.Values must be in the range `[0.0, inf)`.",0.1
,"random_state random_state: int, RandomState instance, default=NoneUsed for shuffling the data, when ``shuffle`` is set to ``True``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",42


### 6. Evalueren

In [87]:
y_pred = model.predict(X_test)

# MSE = meet hoe fout de model zit
mse = mean_squared_error(y_test, y_pred)
# r2 = hoe goed de model de data uitlegt
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("R2:", r2)

MSE: 1497106037.8897483
R2: 0.8132713329821972


### 7. Slim experimenteren
Zie logboek